# Lego Minifig EDA
Exploratory Data Analysis before model comparison (ResNet-50 vs EfficientNet-B0)

Sections:
1. Setup & Data Loading
2. Class Distribution
3. Image Quality Check
4. Class Similarity (sample grids)
5. Metadata Analysis
6. Pixel Statistics
7. Summary & Recommendations

## 1. Setup & Data Loading

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_DIR  = Path(r'C:\Users\ekung\Desktop\KU Lueven\semester2\big data\Assignment\Assignment 2\project')
JSON_PATH    = PROJECT_DIR / 'data' / 'metadata' / 'minifigs.json'
IMAGE_DIR    = PROJECT_DIR / 'data' / 'images'
OUTPUT_DIR   = PROJECT_DIR / 'comparison' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project  : {PROJECT_DIR}')
print(f'JSON     : {JSON_PATH.exists()}')
print(f'Image dir: {IMAGE_DIR.exists()}')

In [ ]:
# Load metadata
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

df = pd.DataFrame(raw)
print(f'Total records : {len(df)}')
print(f'Columns       : {df.columns.tolist()}')
df.head(3)

## 2. Class Distribution

In [ ]:
cat_counts = df['category'].value_counts()
print(f'Total categories : {len(cat_counts)}')
print(f'Largest class    : {cat_counts.index[0]}  ({cat_counts.iloc[0]} images)')
print(f'Smallest class   : {cat_counts.index[-1]}  ({cat_counts.iloc[-1]} images)')
print(f'Imbalance ratio  : {cat_counts.iloc[0] / cat_counts.iloc[-1]:.1f}x')

In [ ]:
# Full category distribution
fig, ax = plt.subplots(figsize=(14, 6))
cat_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axhline(y=100, color='red', linestyle='--', linewidth=1.5, label='MIN_SAMPLES = 100')
ax.set_title('Images per Category (all 122)', fontsize=13)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Images')
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution_all.png')
plt.show()

In [ ]:
# Kept vs dropped categories
MIN_SAMPLES = 100
kept    = cat_counts[cat_counts >= MIN_SAMPLES]
dropped = cat_counts[cat_counts <  MIN_SAMPLES]

print(f'Categories kept    : {len(kept)}  ({kept.sum()} images)')
print(f'Categories dropped : {len(dropped)}  ({dropped.sum()} images)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kept.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Kept Categories (>= {MIN_SAMPLES} images)', fontsize=11)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)

dropped.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title(f'Dropped Categories (< {MIN_SAMPLES} images)', fontsize=11)
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kept_vs_dropped.png')
plt.show()

In [ ]:
# Imbalance within kept categories
fig, ax = plt.subplots(figsize=(12, 4))
kept.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Class Imbalance — Kept 28 Categories', fontsize=12)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Images')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'imbalance_kept.png')
plt.show()

print(f'\nMax / Min ratio in kept categories: {kept.iloc[0] / kept.iloc[-1]:.1f}x')

## 3. Image Quality Check

In [ ]:
# Check image sizes and modes for a sample of 500 images
sample_paths = list((IMAGE_DIR / 'images').glob('*.jpg'))[:500]
print(f'Sampling {len(sample_paths)} images for quality check...')

widths, heights, modes = [], [], []
corrupt = []

for p in sample_paths:
    try:
        img = Image.open(p)
        widths.append(img.width)
        heights.append(img.height)
        modes.append(img.mode)
    except Exception:
        corrupt.append(p.name)

print(f'Corrupt images     : {len(corrupt)}')
print(f'Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}')
print(f'Image modes        : {Counter(modes)}')

In [ ]:
# Width and height distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Count')

axes[1].hist(heights, bins=30, color='salmon', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'image_size_distribution.png')
plt.show()

## 4. Class Similarity — Sample Grids

In [ ]:
# Filter to kept categories only
kept_cats  = kept.index.tolist()
df_kept    = df[df['category'].isin(kept_cats)].copy()

def show_category_grid(df, categories, n_per_cat=4, title='Sample Images'):
    """Show n_per_cat sample images for each category."""
    n_cats = len(categories)
    fig, axes = plt.subplots(n_per_cat, n_cats, figsize=(2.5 * n_cats, 2.5 * n_per_cat))
    if n_per_cat == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(title, fontsize=12, y=1.01)

    for col, cat in enumerate(categories):
        subset  = df[df['category'] == cat]
        samples = subset.sample(n=min(n_per_cat, len(subset)), random_state=42)
        for row, (_, sample) in enumerate(samples.iterrows()):
            img_path = IMAGE_DIR / sample['img_local_path']
            try:
                img = Image.open(img_path).convert('RGB')
                axes[row, col].imshow(img)
            except Exception:
                axes[row, col].text(0.5, 0.5, 'N/A', ha='center', va='center')
            if row == 0:
                axes[row, col].set_title(cat[:15], fontsize=7)
            axes[row, col].axis('off')

    plt.tight_layout()
    return fig

In [ ]:
# Show first 8 kept categories
fig = show_category_grid(df_kept, kept_cats[:8], n_per_cat=3, title='Sample Images — First 8 Categories')
fig.savefig(OUTPUT_DIR / 'sample_grid_1.png', bbox_inches='tight')
plt.show()

In [ ]:
# Show next 8 kept categories
fig = show_category_grid(df_kept, kept_cats[8:16], n_per_cat=3, title='Sample Images — Next 8 Categories')
fig.savefig(OUTPUT_DIR / 'sample_grid_2.png', bbox_inches='tight')
plt.show()

In [ ]:
# Show remaining kept categories
fig = show_category_grid(df_kept, kept_cats[16:], n_per_cat=3, title='Sample Images — Remaining Categories')
fig.savefig(OUTPUT_DIR / 'sample_grid_3.png', bbox_inches='tight')
plt.show()

## 5. Metadata Analysis

In [ ]:
# Missing values overview
print('Missing values per column:')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(missing_df[missing_df['missing'] > 0].to_string())

In [ ]:
# Year distribution
df['year_released'] = pd.to_numeric(df['year_released'], errors='coerce')

fig, ax = plt.subplots(figsize=(12, 4))
df['year_released'].dropna().astype(int).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='white'
)
ax.set_title('Minifigs Released per Year', fontsize=12)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'year_distribution.png')
plt.show()

In [ ]:
# Average year per category (kept only)
year_by_cat = df_kept.groupby('category')['year_released'].mean().sort_values()

fig, ax = plt.subplots(figsize=(12, 4))
year_by_cat.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average Year Released per Category', fontsize=12)
ax.set_xlabel('Category')
ax.set_ylabel('Avg Year')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'year_by_category.png')
plt.show()

In [ ]:
# Themes analysis — how many themes does each minifig belong to?
df['n_themes'] = df['themes'].apply(lambda x: len(x) if isinstance(x, list) else 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['n_themes'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Number of Themes per Minifig')
axes[0].set_xlabel('Number of Themes')
axes[0].set_ylabel('Count')

# Top 15 most common themes
all_themes = [t for themes in df['themes'].dropna() for t in themes]
theme_counts = Counter(all_themes)
top_themes = pd.Series(theme_counts).nlargest(15)
top_themes.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Top 15 Most Common Themes')
axes[1].set_xlabel('Theme')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'themes_analysis.png')
plt.show()

## 6. Pixel Statistics

In [ ]:
# Compute average brightness per category (sample 20 images per category)
print('Computing brightness per category...')
brightness_by_cat = {}

for cat in kept_cats:
    subset  = df_kept[df_kept['category'] == cat].sample(n=min(20, len(df_kept[df_kept['category'] == cat])), random_state=42)
    values  = []
    for _, row in subset.iterrows():
        try:
            img = Image.open(IMAGE_DIR / row['img_local_path']).convert('L')  # grayscale
            values.append(np.array(img).mean())
        except Exception:
            pass
    if values:
        brightness_by_cat[cat] = np.mean(values)

brightness_series = pd.Series(brightness_by_cat).sort_values()

fig, ax = plt.subplots(figsize=(12, 4))
brightness_series.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Average Image Brightness per Category', fontsize=12)
ax.set_xlabel('Category')
ax.set_ylabel('Mean Pixel Value (0=black, 255=white)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'brightness_by_category.png')
plt.show()

print(f'Brightest category : {brightness_series.index[-1]}  ({brightness_series.iloc[-1]:.1f})')
print(f'Darkest category   : {brightness_series.index[0]}   ({brightness_series.iloc[0]:.1f})')

In [ ]:
# Overall pixel mean and std (sample 200 images)
print('Computing overall pixel statistics...')
sample_imgs = df_kept.sample(n=200, random_state=42)
r_vals, g_vals, b_vals = [], [], []

for _, row in sample_imgs.iterrows():
    try:
        img = np.array(Image.open(IMAGE_DIR / row['img_local_path']).convert('RGB')) / 255.0
        r_vals.append(img[:, :, 0].mean())
        g_vals.append(img[:, :, 1].mean())
        b_vals.append(img[:, :, 2].mean())
    except Exception:
        pass

print(f'Mean  R/G/B : {np.mean(r_vals):.3f} / {np.mean(g_vals):.3f} / {np.mean(b_vals):.3f}')
print(f'Std   R/G/B : {np.std(r_vals):.3f}  / {np.std(g_vals):.3f}  / {np.std(b_vals):.3f}')
print(f'ImageNet    : 0.485 / 0.456 / 0.406  (for comparison)')

## 7. Summary & Recommendations

In [ ]:
print('=' * 55)
print('EDA SUMMARY')
print('=' * 55)
print(f'Total records         : {len(df)}')
print(f'Total categories      : {len(cat_counts)}')
print(f'Categories kept (>=100): {len(kept)}')
print(f'Categories dropped    : {len(dropped)}')
print(f'Images in kept cats   : {kept.sum()}')
print(f'Imbalance ratio (kept): {kept.iloc[0] / kept.iloc[-1]:.1f}x')
print(f'Missing character_name: {df["character_name"].isnull().sum()}')
print('=' * 55)
print('\nRecommendations:')
print('  1. Use WeightedRandomSampler — class imbalance is significant')
print('  2. ImageNet normalisation is appropriate — pixel stats are similar')
print('  3. All images are similar size — no special resizing needed')
print('  4. Metadata (year, themes) could be used in a future multimodal model')
print('  5. Visually similar categories may confuse the model — check confusion matrix after training')